<a href="https://colab.research.google.com/github/diegoo7am/Curso-Intro-Python/blob/main/Modulo_02_Aplicaciones_Financieras/2_5_analisis_fundamental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[⬅ Volver al índice](https://colab.research.google.com/github/diegoo7am/Curso-Intro-Python/blob/main/indice.ipynb)

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Análisis fundamental.

## Extracción de la información contable y financiera

Yahoo Finance permite acceder a toda la información que tiene publicada en su página web sobre una empresa. El acceso se realiza mediante la **programación orientada a objetos (POO)**:

`empresa = yf.Ticker("ITX.MC")`

`yf.Ticker` es una clase que es como una plantilla y `empresa` es una **instancia u objeto** de esa clase, es decir, un caso concreto creado con la plantilla.

In [ ]:
empresa = yf.Ticker("ITX.MC")
empresa

`empresa` no es simplemente una variable que contiene un número o una lista. Es un objeto bastante más complejo que contiene:

- Datos y atributos
- Funciones asociadas al propio objeto que denominamos métodos

In [ ]:
print(type(empresa))

In [ ]:
dir(empresa)

## CUENTA DE RESULTADOS

In [ ]:
resultados=empresa.income_stmt
resultados

In [ ]:
type(empresa.income_stmt)

In [ ]:
resultados.index

In [ ]:
orden = [
    "Total Revenue",
    "Cost Of Revenue",
    "Gross Profit",
    "Operating Expense",
    "Operating Income",
    "Interest Expense",
    "Pretax Income",
    "Tax Provision",
    "Net Income"
]

Voy a crear una list comprehension que es una nueva lista que voy a llamar `orden_disponible` en el que guardaremos únicamente las partidas de la lista `orden` que está disponible en `resultados.index`

In [ ]:
orden_disponible = [
    partida for partida in orden
    if partida in resultados.index
]
resultados= resultados.loc[orden_disponible] # seleccionamos las filas que necesitamos.
resultados

In [ ]:
# Utilizamos un diccionario para proceder a la traducción de las partidas contables
traduccion = {
    "Total Revenue": "Ingresos",
    "Cost Of Revenue": "Coste de ventas",
    "Gross Profit": "Margen bruto",
    "Operating Expense": "Gastos operativos",
    "Operating Income": "Resultado operativo",
    "Interest Expense": "Gastos financieros",
    "Pretax Income": "Resultado antes de impuestos",
    "Tax Provision": "Impuestos",
    "Net Income": "Beneficio neto"
}

In [ ]:
resultados_es = resultados.rename(index=traduccion)
resultados_es

**EJERCICIO** Extrae el balance con la siguiente estructura y tradúcelo al español

In [ ]:
orden_balance = [
    "Cash Cash Equivalents And Short Term Investments",
    "Accounts Receivable",
    "Inventory",
    "Current Assets",

    "Net PPE",
    "Goodwill And Other Intangible Assets",
    "Total Assets",

    "Current Liabilities",
    "Long Term Debt",
    "Total Liabilities Net Minority Interest",

    "Stockholders Equity"
]

In [ ]:
balance = empresa.balance_sheet
balance

In [ ]:
# Escribe aquí tu código

**EJERCICIO** Extrae la información del flujo de caja con la siguiente estructura y tradúcela al español.

In [ ]:
orden_flujos = [
    "Net Income From Continuing Operations",
    "Depreciation And Amortization",
    "Change In Working Capital",
    "Operating Cash Flow",

    "Capital Expenditure",
    "Investing Cash Flow",

    "Issuance Of Debt",
    "Repayment Of Debt",
    "Repurchase Of Capital Stock",
    "Cash Dividends Paid",
    "Financing Cash Flow",

    "Changes In Cash"
]

In [ ]:
cashflow = empresa.cashflow
cashflow

In [ ]:
# Escribe aquí tu código

# ANÁLISIS DE LA RENTABILIDAD

## Rentabilidad económica

$$ROA=\frac{EBIT}{Activo Total}=\frac{EBIT}{Ventas}\times\frac{Ventas}{Activo Total}$$

Donde $\frac{EBIT}{Ventas}$ es el **margen** y $\frac{Ventas}{Activo Total}$ es la **rotación** del activo.

In [ ]:
tickers = [
    "ITX.MC",   # Inditex
    "ADS.DE",   # Adidas
    "MC.PA",    # LVMH
    "NKE",      # Nike
    "9983.T",   # Fast Retailing / Uniqlo
    "BOSS.DE"   # Hugo Boss
]

Como los estados financieros tienen una columna por ejercicio, vamos a crear una función que devuelva el dato del **ejercicio más reciente** de una partida (o `NaN` si la empresa no la publica):

In [ ]:
def ultimo_valor(estado: pd.DataFrame, partida: str) -> float:
    """Devuelve el valor más reciente de una partida de un estado financiero.

    Args:
        estado (pd.DataFrame): Estado financiero de yfinance (una columna por ejercicio)
        partida (str): Nombre de la partida (fila)

    Returns:
        float: Valor del último ejercicio disponible o NaN si no existe
    """
    if partida not in estado.index:
        return np.nan
    serie = estado.loc[partida]
    serie = serie[sorted(serie.index, reverse=True)].dropna()  # del ejercicio más reciente al más antiguo
    return serie.iloc[0] if len(serie) > 0 else np.nan

**EJERCICIO** Estima el margen, la rotacion y la rentabilidad economica (ROA) de las empresas. Calcula en primer lugar la de Inditex "ITX.MC"

In [ ]:
# Escribe aquí tu código

**EJERCICIO** Ahora lo calculamos para el resto de empresas del sector para poder compararlas.

In [ ]:
# Escribe aquí tu código

Descomponemos la rotación
$$\frac{Ventas}{\text{Activo Total}}=\left\{
\begin{array}{l}
\frac{Ventas}{\text{Activo Corriente}} \\
\frac{Ventas}{\text{Activo No Corriente}}\end{array}
\right.$$

**EJERCICIO** Calcula ambas rotaciones para todas las empresas.

In [ ]:
# Escribe aquí tu código

# ROE

## Rentabilidad Financiera
$$ROE=\frac{\text{Beneficio Neto}}{Patrimonio}$$

## Apalancamiento Financiero

$$ROC=\frac{\text{Beneficio sin Intereses y Después de Impuestos}}{Activo Total}=\frac{EBIT\times(1-\tau)}{Activo Total}$$

$$ROE=ROC+\left(ROC-k_d(1-\tau)\right)\times \frac{Deuda}{Patrimonio}$$

$$k_d=\frac{Intereses}{Deuda}$$


$$\tau=\frac{Impuestos}{\text{Beneficio Antes de Impuestos}}$$

**EJERCICIO** Calcula el ROE de cada empresa y compáralo con su descomposición en ROC y efecto apalancamiento. ¿En qué empresas el endeudamiento aumenta la rentabilidad del accionista?

In [ ]:
# Escribe aquí tu código

# RATIOS DE MERCADO

$$Price-to-book=\frac{\text{Capitalización Bursátil}}{\text{Patrimonio Neto}}$$


$$PER=\frac{Precio}{\text{Beneficio por Acción}}=\frac{\text{Capitalización Bursátil}}{\text{Beneficio Neto}}$$

In [ ]:
empresa = yf.Ticker("ITX.MC")

price_to_book = empresa.info.get("priceToBook")
per = empresa.info.get("trailingPE")

print(price_to_book,per)

**EJERCICIO** Compara el PER y el Price to Book entre todas las empresas

In [ ]:
# Escribe aquí tu código

**EJERCICIO** Analiza si los fundamentales estimados explican el comportamiento de la rentabilidad de las empresas analizadas durante el último año.

In [ ]:
# Escribe aquí tu código